# L07 · Detectar problemas antes de limpiar

| | |
|---|---|
| **Qué vas a aprender** | Perfilar la calidad de un dataset antes de tocarlo |
| **Cuánto tiempo** | 60 min |
| **Datasets** | los 8 ficheros de `datos/` |

> **Al terminar vas a poder** decir, con números, cuán sucio es un dataset. Y
> justificar cada limpieza que hagas en vez de aplicarla a ojo.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

## El concepto

Limpiar sin medir antes es hacer las cosas a ciegas: quitas cosas sin saber qué estás
quitando ni cuánto valen. El orden correcto es **perfilar → decidir → limpiar → volver
a perfilar**.

Y hay una trampa conceptual que hay que tener clara ya: `isna()` solo ve los nulos que
pandas representa como `NaN`/`None`. Los "falsos nulos" — un `"N/A"` escrito por una
persona, un `"-"`, un `0` en una columna de edad — **no existen para `isna()`**, y
son justo los que más problemas dan.

## Manos a la obra

In [ ]:
fin = pd.read_csv(
    DATOS / "finanzas_conciliacion.csv", sep=";", encoding="latin-1"
)
print("shape:", fin.shape)
mostrar(fin, 5, "EXTRACTO BANCARIO (crudo)")

### Nulos reales

In [ ]:
print("nulos reales por columna:")
print(fin.isna().sum()[lambda s: s > 0])

### Falsos nulos: los que `isna()` no ve

En `conciliado` hay `"S"`, `"N"`, celdas vacías, `"sí"` y `"no"`. Al leer el CSV
las celdas vacías ya son `NaN`: `isna()` las ve. `"sí"` y `"no"` **no son nulos**,
son la misma respuesta escrita de otra forma.

Los falsos nulos de este proyecto están en la encuesta: `"ns/nc"`, `"-"`, `"N/S"`
y `"NN/NN"`. `isna()` no los ve, y hay que compararlos sin mirar las mayúsculas.

In [ ]:
from herramientas import contar_falsos_nulos
print("valores en 'conciliado' (sí/no no son nulos):")
print(fin["conciliado"].value_counts(dropna=False).to_string())
print("falsos nulos en conciliado:", int(contar_falsos_nulos(fin["conciliado"]).sum()))

enc = pd.read_csv(DATOS / "encuestas_satisfaccion.csv")
print()
print("falsos nulos en satisfaccion:")
print(contar_falsos_nulos(enc["satisfaccion"]).to_string())
print("falsos nulos en comentario (NN/NN):")
print(contar_falsos_nulos(enc["comentario"]).to_string())

### La columna `importe`: cinco formatos en la misma columna

In [ ]:
importes = fin["importe"].dropna().astype(str)
muestra = importes.sample(10, random_state=1).tolist()
for s in muestra:
    print(f"  {s!r}")

print()
print("todos los que NO son float directo tienen algo raro:")
raros = importes[~importes.str.match(r"^-?\d+\.\d+$")]
print(f"  {len(raros):,} de {len(importes):,} son 'raros' "
      f"({len(raros) / len(importes) * 100:.1f}%)")

### Duplicados: distinguir el duplicado de negocio

In [ ]:
print("filas exactamente duplicadas:", fin.duplicated().sum())
print()
# una referencia repetida NO es un duplicado: son dos movimientos distintos
print("referencias repetidas:", fin["referencia"].duplicated().sum())
print()
top = fin["referencia"].value_counts().head(5)
mostrar(top, 5, "REFERENCIAS MÁS REPETIDAS")

## ⚠️ La trampa

### `drop_duplicates(subset="pedido")` destruye las líneas

`drop_duplicates()` sin `subset` solo tira filas **idénticas en todas las columnas**.
En una tabla de líneas, cada línea es distinta, así que no tira nada.

El borrado caro es `subset="pedido"`: te quedas con una línea por pedido y el
importe del resto desaparece, sin un warning.

In [ ]:
lineas_pedido = pd.DataFrame({
    "pedido": [1, 1, 1, 2, 2],
    "linea": ["A", "B", "C", "A", "B"],
    "importe": [10.0, 20.0, 30.0, 5.0, 6.0],
})

# Las cinco filas son distintas: esto no quita ninguna.
sin_subset = lineas_pedido.drop_duplicates()
# Esto sí: una fila por pedido. Se pierde el histórico de líneas.
una_por_pedido = lineas_pedido.drop_duplicates(subset="pedido")

print("original          :", len(lineas_pedido), "->", lineas_pedido["importe"].sum())
print("sin subset        :", len(sin_subset), "->", sin_subset["importe"].sum())
print("subset='pedido'   :", len(una_por_pedido), "->", una_por_pedido["importe"].sum(),
      "  <-- se han perdido lineas")

El total pasa de 71 a 15 en la llamada con `subset="pedido"`. La llamada
sin `subset` sigue en 71, porque ninguna fila estaba repetida entera.

## 🧪 Tu turno

- **Ejercicio 1** — Para los 8 ficheros de `datos/`, calcula el nº de nulos reales y el
  de falsos nulos por columna, y monta una tabla comparativa.
- **Ejercicio 2** — ¿En qué fichero hay más filas 100% duplicadas? ¿Son duplicados de
  negocio o de importación?
- **Ejercicio 3** — Detecta el patrón del `importe`: agrupa los valores por su "forma"
  (con paréntesis, con "EUR", con coma, con punto) y cuenta cada forma.

## 📦 Workbook integrador: informe de calidad de datos

**Contexto.** Antes de montar tu dashboard, el equipo de datos quiere saber si se puede
confiar en estas fuentes. Te piden un informe de calidad de los 8 ficheros.

**Tu trabajo.** Escribir una función `perfilar(df)` que devuelva, por columna:

- nº y % de nulos reales
- nº y % de falsos nulos (valores de la lista de `herramientas.FALSOS_NULOS`)
- nº de valores distintos (cardinalidad)
- nº de filas duplicadas del fichero (filas idénticas en todas las columnas)
- el dtype

Y aplicarla a los 8 ficheros, exportando **un único CSV consolidado** con una columna
que identifique el fichero de origen.

Este perfil mide nulos. No mide nombres duplicados ni precios ×100. El CRM y el
catálogo pueden salir "limpios" aquí y seguir siendo los ficheros más sucios
por otro motivo.

**Restricción importante.** El informe tiene que poder ejecutarse sobre un dataset
**nuevo** sin cambiar el código. Nada de nombres de columna escritos a mano.

In [ ]:
FUENTES = (
    sorted(DATOS.glob("*.csv"))
    + sorted(DATOS.glob("*.tsv"))
    + sorted(DATOS.glob("*.xlsx"))
)


def perfilar(df: pd.DataFrame, origen: str) -> pd.DataFrame:
    """Una fila por columna, con sus métricas de calidad."""
    filas = []
    n = len(df)
    filas_dup = int(df.duplicated().sum())
    for col in df.columns:
        s = df[col]
        falsos = int(contar_falsos_nulos(s).sum())
        filas.append({
            "fichero": origen,
            "columna": col,
            "dtype": str(s.dtype),
            "filas": n,
            "filas_duplicadas": filas_dup,
            "nulos_reales": int(s.isna().sum()),
            "%_nulos": round(s.isna().sum() / n * 100, 2) if n else 0.0,
            "falsos_nulos": falsos,
            "%_falsos_nulos": round(falsos / n * 100, 2) if n else 0.0,
            "unicos": int(s.nunique(dropna=True)),
            "cardinalidad_%": round(s.nunique(dropna=True) / n * 100, 1) if n else 0.0,
        })
    return pd.DataFrame(filas)


print("perfilando", len(FUENTES), "ficheros...")
informe = pd.concat(
    [perfilar(cargar(Path(f)), Path(f).name) for f in FUENTES],
    ignore_index=True,
)

print("columnas del informe:", informe.shape)
mostrar(informe.sort_values("%_falsos_nulos", ascending=False), 12,
        "LAS COLUMNAS MÁS SUCIAS")

In [ ]:
# resumen por fichero: qué tan sucio es cada uno
resumen = (informe.groupby("fichero")
                  .agg(columnas=("columna", "size"),
                       filas=("filas", "first"),
                       nulos=("nulos_reales", "sum"),
                       falsos=("falsos_nulos", "sum"))
                  .reset_index())
resumen["%_sucio"] = ((resumen["nulos"] + resumen["falsos"]) /
                      (resumen["filas"] * resumen["columnas"]) * 100).round(2)

mostrar(resumen.sort_values("%_sucio", ascending=False), titulo="GRADO DE SUCIEDAD POR FICHERO")

In [ ]:
ruta_informe = guardar_csv(informe, "L07_informe_calidad")
ruta_resumen = guardar_csv(resumen, "L07_resumen_suciedad")
resumen_entrega(ruta_informe, ruta_resumen)

### Criterios de aceptación

1. `perfilar()` corre sobre los 8 ficheros sin tocar una sola línea entre uno y otro.
2. `informe` tiene una fila por (fichero, columna), y su nº de filas es exactamente la
   suma de columnas de los 8 ficheros.
3. La columna `%_falsos_nulos` es **mayor que 0** en la encuesta (`ns/nc`, `-`, `NN/NN`).
4. `resumen` ordena por nulos. La encuesta y el extracto bancario quedan por encima
   del CRM y del catálogo: su suciedad (nombres y precios) no es un nulo.
5. `filas_duplicadas` del CRM es 400: las copias exactas sembradas a propósito.
6. Se exportan los dos CSV en `salida/`.